In [21]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score
import xgboost as xgb
import joblib
import os

ml_data = pd.read_csv('../data/processed/ml_data_ready.csv')

train_df, test_df = train_test_split(ml_data, test_size=0.2, random_state=42)
club_stats = train_df.groupby('current_club_id').agg(
    club_total_value = ('market_value_in_m', 'sum'),
    club_player_count = ('market_value_in_m', 'count')
).reset_index()
train_df = train_df.merge(
    club_stats,
    on = 'current_club_id',
    how = 'left'
)
train_df['teammates_mean_value'] = (
    (train_df['club_total_value'] - train_df['market_value_in_m']) / (train_df['club_player_count'] - 1)
).fillna(0)
# claculam pentru test_df acum
club_stats['club_mean_value'] = (club_stats['club_total_value'] / club_stats['club_player_count']).fillna(0)
test_df = test_df.merge(
    club_stats[['club_mean_value', 'current_club_id']],
    on = 'current_club_id',
    how = 'left'
)
# daca gasim un jucator cu un club care nu a fost in train, punem media globala
global_mean = train_df['market_value_in_m'].mean()
test_df['teammates_mean_value'] = test_df['club_mean_value'].fillna(global_mean)

# Encodare
y_train = train_df['market_value_in_m']
y_test = test_df['market_value_in_m']
drop_train = ['market_value_in_m', 'player_id', 'current_club_id', 'club_total_value', 'club_player_count']
drop_test = ['market_value_in_m', 'player_id', 'current_club_id', 'club_mean_value']
X_train_raw = train_df.drop(columns=[col for col in drop_train if col in train_df.columns])
X_test_raw = test_df.drop(columns=[col for col in drop_test if col in test_df.columns])
# Econdam textul cu one-hot encoding pe train si test
X_train = pd.get_dummies(X_train_raw, drop_first=True)
X_test = pd.get_dummies(X_test_raw, drop_first=True)
# Aliniem coloanele pt a ne asigura ca x_train si x_test sunt 100% identice
X_train, X_test = X_train.align(X_test, join='left', axis=1, fill_value=0)
print(f"Randuri pentru antrenare (Train): {len(X_train)}")
print(f"Randuri pentru testare (Test): {len(X_test)}")


# Vom folosi random forest pentru algoritmul de tip regresie. Practic luam 100 de "scouteri" si fiecare face o evaluare si la final facem media evaluarilor scouterilor.

regressor = RandomForestRegressor(
    n_estimators = 100,
    random_state = 42, # acest state se asigura totusi ca rezultatele nu depind de generarea aleatoare
    n_jobs = -1 # ne folosim de toate thread-urile procesului pentru a rula mai repede
)
regressor.fit(X_train, y_train)
y_pred = regressor.predict(X_test)

# evaluare model => r2 score = nota modelului, ne spune cat de bine a prezis
# mean absolute error => in medie ne spune cu cate milioane de euro da pe langa modelul

error = mean_absolute_error(y_test, y_pred)
r2score = r2_score(y_test, y_pred)
print(error, r2score)



errors_df = pd.DataFrame(
    {
        'real' : y_test,
        'predicted' : y_pred,
        'error': abs(y_test - y_pred)
    }
).sort_values('error', ascending = False)

# xgboost

xgb_model = xgb.XGBRegressor(
    n_estimators = 200,
    learning_rate = 0.05,
    max_depth = 6,
    random_state = 42,
    subsample = 0.8,
    n_jobs = -1
)

xgb_model.fit(X_train, y_train)
y_pred_xgb = xgb_model.predict(X_test)
error_xgb = mean_absolute_error(y_test, y_pred_xgb)
r2_score_xgb = r2_score(y_test, y_pred_xgb)
print(error_xgb, r2_score_xgb)


# Analiza
feature_importances = pd.Series(data = xgb_model.feature_importances_, index = X_train.columns).sort_values(ascending = False)
feature_importances.head(50)


# Salvare Model pe disc
xgb_model.save_model("../models/xgb_model.json")
joblib.dump(X_train.columns.tolist(), "../models/feature_columns.joblib")



Randuri pentru antrenare (Train): 11041
Randuri pentru testare (Test): 2761
1.6686250452734515 0.8237343668029444
1.6483168630966982 0.8341636514313782


['../models/feature_columns.joblib']

In [25]:
import matplotlib.pyplot as plt
import seaborn as sns
import os


# creating 'images' folder in project root
os.makedirs('../images', exist_ok=True)

print("Generating graph... Actual vs Predicted...")
# graph: Actual vs Predicted
plt.figure(figsize=(10, 6))
sns.scatterplot(x=y_test, y=y_pred_xgb, alpha=0.5, color='#1f77b4')
# red line = ideal
plt.plot([0, y_test.max()], [0, y_test.max()], color='red', linestyle='--', linewidth=2)
plt.title('Real Market Value vs Predicted Market Value (XGBoost)', fontsize=14)
plt.xlabel('Real Value (Millions €)', fontsize=12)
plt.ylabel('Predicted Value (Millions €)', fontsize=12)
plt.grid(True, alpha=0.3)
# saving picture
plt.savefig('../images/actual_vs_predicted.png', bbox_inches='tight')
plt.close()

print("Generam graficul Feature Importance...")
# Graph : Feature Importance (Top 15)
plt.figure(figsize=(12, 8))
top_features = feature_importances.head(15)
sns.barplot(x=top_features.values, y=top_features.index, palette='viridis')
plt.title('Top 15 Most Important Features for Player Valuation', fontsize=14)
plt.xlabel('F-Score (Importance)', fontsize=12)
plt.ylabel('Feature', fontsize=12)
# saving picture
plt.savefig('../images/feature_importance.png', bbox_inches='tight')
plt.close()

print("✅ Graphs were succesfully saved!")

Generating graph... Actual vs Predicted...
Generam graficul Feature Importance...


C:\Users\zagan\AppData\Local\Temp\ipykernel_16040\3145121250.py:27: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(x=top_features.values, y=top_features.index, palette='viridis')


✅ Graphs were succesfully saved!
